## Import

In [ ]:
import glob
import json
import math
import os
import random
import shutil as sh
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import cv2 as cv
import numpy as np
import pandas as pd
import rootutils
from joblib import Parallel, delayed
from numpy.typing import NDArray
from PIL import Image, ImageDraw, ImageFilter
from rich import print

rootutils.setup_root(".", indicator=".project-root", pythonpath=True, cwd=True)


@dataclass
class SyntheticBg:
    path: Path
    circle_like: bool = False


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
RNG = np.random.default_rng(seed=SEED)
# torch.manual_seed(SEED)
# torch.cuda.manual_seed(SEED)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = True

DATA_DIR = Path("data")
DATASET_DIR = DATA_DIR / "dataset"

SRC_DIR = Path("data/datasets/seeds_rgb")
SRC_IMAGES_WITH_BG_DIR = SRC_DIR / "images_with_bg"
SRC_IMAGES_DIR = SRC_DIR / "images"
SRC_MASKS_DIR = SRC_DIR / "masks"
DST_DIR = DATASET_DIR / "seed_detect"

DST_IMAGES_DIR = DST_DIR / "images"
DST_LABELS_DIR = DST_DIR / "labels"
DST_ANNO_DIR = DST_DIR / "annotations"
DST_MASKS_DIR = DST_DIR / "masks"
DST_MASKS_IMG_DIR = DST_DIR / "masks_img"
SYNTHETIC_BG_DIR = Path("notebooks/synthetic_bgs")

DST_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
DST_LABELS_DIR.mkdir(parents=True, exist_ok=True)
DST_MASKS_DIR.mkdir(parents=True, exist_ok=True)
DST_MASKS_IMG_DIR.mkdir(parents=True, exist_ok=True)

DS_INFO = {
    "description": "OpenSeed-LZU object detection dataset",
    "author": "",
    "url": "",
}


P_TRAIN = 0.8
P_VAL = 0.1
P_TEST = 0.1
NUM_RECTS_MIN = 1
NUM_RECTS_MAX = 100
DATASET_SIZE = 20000
N_JOBS = 30


def random_color():
    def r():
        return random.randint(0, 255)

    return f"#{r():02X}{r():02X}{r():02X}"


syn_bgs = [
    SyntheticBg(SYNTHETIC_BG_DIR / "1.png", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "2.png", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "3.png", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "4.jpg", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "5.png", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "6.png", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "7.png", True),
    SyntheticBg(SYNTHETIC_BG_DIR / "8.png", True),
    SyntheticBg(SYNTHETIC_BG_DIR / "9.png", True),
    SyntheticBg(SYNTHETIC_BG_DIR / "10.png", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "11.png", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "12.png", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "13.png", True),
    SyntheticBg(SYNTHETIC_BG_DIR / "14.png", True),
    SyntheticBg(SYNTHETIC_BG_DIR / "15.png", True),
    SyntheticBg(SYNTHETIC_BG_DIR / "16.png", True),
    SyntheticBg(SYNTHETIC_BG_DIR / "17.png", True),
    SyntheticBg(SYNTHETIC_BG_DIR / "18.png", True),
    SyntheticBg(SYNTHETIC_BG_DIR / "19.jpg", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "20.jpg", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "21.jpg", False),
    SyntheticBg(SYNTHETIC_BG_DIR / "22.jpg", False),
]
# ensure no duplicates
rand_colors = list({random_color() for _ in range(2000)})[:1000]
SYNTHETIC_BACKGROUNDS: list[str | SyntheticBg] = [
    "#4B4E55",
    "#2E2735",
    "#48454E",
    "#AAA7A5",
    "#595257",
    "#292538",
    "#FFFFFF",
    "#000000",
    # *rand_colors,
    *syn_bgs,
]
del syn_bgs

# get image paths and labels
with open(SRC_DIR / "cls_train_656_rgb.json", "r") as fin:
    cls_train = json.load(fin)
with open(SRC_DIR / "cls_val_656_rgb.json", "r") as fin:
    cls_val = json.load(fin)
with open(SRC_DIR / "cls_test_656_rgb.json", "r") as fin:
    cls_test = json.load(fin)
ALL_IMAGES: list[str] = cls_train["images"] + cls_val["images"] + cls_test["images"]
ALL_LABELS: list[int] = cls_train["labels"] + cls_val["labels"] + cls_test["labels"]
ALL_CLASSES: list[str] = cls_train["classes"]
NUM_IMAGES = len(ALL_IMAGES)

# generate original train, val, test datasets
# pre-split to traini val test
_idx = np.arange(NUM_IMAGES)
np.random.shuffle(_idx)
_tmp = [ALL_IMAGES[i] for i in _idx]
_tmp_labels = [ALL_LABELS[i] for i in _idx]
TRAIN_IMAGES = _tmp[: int(P_TRAIN * len(_tmp))]
VAL_IMAGES = _tmp[int(P_TRAIN * len(_tmp)) : int((P_TRAIN + P_VAL) * len(_tmp))]
TEST_IMAGES = _tmp[int((P_TRAIN + P_VAL) * len(_tmp)) :]
TRAIN_LABELS = _tmp_labels[: int(P_TRAIN * len(_tmp_labels))]
VAL_LABELS = _tmp_labels[int(P_TRAIN * len(_tmp_labels)) : int((P_TRAIN + P_VAL) * len(_tmp_labels))]
TEST_LABELS = _tmp_labels[int((P_TRAIN + P_VAL) * len(_tmp_labels)) :]
del cls_train, cls_val, cls_test, _idx, _tmp, _tmp_labels


# calculate frequencies of labels, used for over sampling
# inverse probability weighting (IPW), more frequent labels have smaller sampling probability
# less frequent labels have larger sampling probability
def sampling_p_of(label_set: list[int]):
    cls_labels_prob = {}
    for label in label_set:
        cls_labels_prob[label] = cls_labels_prob.get(label, 0) + 1
    cls_labels_prob = {label: count / len(label_set) for label, count in cls_labels_prob.items()}
    class_sampling_p = np.array(
        [1 / cls_labels_prob[label] for label in label_set],
        dtype=np.float32,
    )
    class_sampling_p /= class_sampling_p.sum()
    return class_sampling_p


TRAIN_SAMPLING_P = sampling_p_of(TRAIN_LABELS)
VAL_SAMPLING_P = sampling_p_of(VAL_LABELS)
TEST_SAMPLING_P = sampling_p_of(TEST_LABELS)

MAP_CLASS_ID = {ALL_CLASSES[i]: i for i in range(len(ALL_CLASSES))}
names_map = pd.read_csv(DATA_DIR / "names_map_sorted_filtered_unique.csv")
MAP_NAME_CN_LA = {row[1]: row[2] for row in names_map.values if row[2] in ALL_CLASSES}
del names_map


categories = [
    "Atriplex centralasiatica",
    "Festuca sinensis",
    "Corethrodendron fruticosum",
    "Amaranthus retroflexus",
    "Carex setschwanensis",
    "Swertia tetraptera",
    "Cleistogenes songorica",
    "Atriplex sibirica",
    "Potentilla multifida",
    "Plantago major",
    "Potentilla chinensis",
    "Elsholtzia densa",
    "Bupleurum smithii var. parvifolium",
    "Euphrasia pectinata",
    "Delphinium kamaonense var. glabrescens",
    "Ixeris chinensis",
    "Elymus dahuricus",
    "Tournefortia sibirica",
    "Gueldenstaedtia verna",
    "Elymus sibiricus",
    "Lespedeza bicolor",
    "Melilotus suaveolens",
    "Capsella bursa-pastoris",
    "Carum carvi",
    "Prunus mongolica",
    "Anemone rivularis var. flore-minore",
    "Carex tibetikobresia",
    "Semenovia malcolmii",
    "Leymus secalinus",
    "Daucus carota",
    "Aster altaicus",
    "Zygophyllum xanthoxylum",
    "Festuca elata",
    "Lappula myosotis",
    "Datura stramonium",
    "Caragana halodendron",
    "Elymus kamoji",
    "Achnatherum inebrians",
    "Elymus nutans",
    "Sorghum bicolor × sudanense",
    "Nitraria tangutorum",
    "Sophora alopecuroides",
    "Nitraria sibirica",
    "Rumex crispus",
    "Rumex acetosa",
    "Stipa grandis",
    "Bidens parviflora",
    "Incarvillea sinensis var. przewalskii",
    "Reaumuria songarica",
    "Elymus dahuricus var. cylindricus",
    "Caragana liouana",
    "Elaeagnus angustifolia",
    "Vicia sativa subsp. nigra",
    "Poa araratica",
    "Caryopteris mongholica",
    "Bromus japonicus",
    "Agropyron cristatum",
    "Brachypodium sylvaticum",
    "Onobrychis viciifolia",
    "Corethrodendron scoparium",
    "Malva cathayensis",
    "Lycium chinense",
    "Chenopodium album",
    "Apocynum venetum",
    "Apocynum pictum",
    "Galeopsis bifida",
    "Caryopteris incana",
    "Bassia scoparia",
    "Plantago depressa",
    "Descurainia sophia",
    "Haloxylon ammodendron",
    "Eragrostis pilosa",
    "Ligularia sagitta",
    "Trifolium pratense",
    "Thlaspi arvense",
    "Corispermum patelliforme",
    "Hibiscus trionum",
    "Saposhnikovia divaricata",
    "Gymnaconitum gymnandrum",
    "Carex moorcroftii",
    "Rhaponticum repens",
    "Peganum harmala",
    "Melilotus officinalis",
    "Tamarix chinensis",
    "Iris lactea",
    "Sorghum bicolor",
    "Ammopiptanthus mongolicus",
    "Sphaerophysa salsula",
    "Vicia sativa",
    "Vicia villosa",
    "Atriplex canescens",
    "Pennisetum alopecuroides",
    "Pedicularis semitorta",
    "Avena sativa",
    "Stipa breviflora",
    "Chenopodiastrum hybridum",
    "Halenia elliptica",
    "Malva verticillata var. rafiqii",
    "Suaeda prostrata",
    "Astragalus membranaceus",
    "Rumex patientia",
    "Bromus inermis",
    "Nitraria sphaerocarpa",
    "Stipa bungeana",
    "Eruca vesicaria subsp. sativa",
    "Artemisia desertorum",
    "Oxybasis glauca",
    "Persicaria lapathifolia",
    "Grubovia dasyphylla",
    "Leonurus japonicus",
    "Astragalus melilotoides",
    "Plantago minuta",
    "Hyoscyamus niger",
    "Trollius chinensis",
    "Allium ramosum",
    "Sanguisorba officinalis",
    "Kalidium cuspidatum var. sinicum",
    "Sonchus oleraceus",
    "Ligularia virgaurea",
    "Setaria viridis",
    "Sibbaldianthe bifurca",
    "Potentilla supina",
    "Potentilla acaulis",
    "Caragana stenophylla",
    "Carex atrofusca subsp. minor",
    "Pleurospermum uralense",
    "Adenophora stenanthina",
    "Gentianopsis paludosa",
    "Astragalus laxmannii",
    "Pedicularis sylvatica",
    "Corydalis adunca",
    "Anthriscus sylvestris",
    "Medicago archiducis-nicolai",
    "Setaria pumila",
    "Limonium bicolor",
    "Erodium stephanianum",
    "Oxytropis bicolor",
    "Suaeda glauca",
    "Festuca rubra subsp. arctica",
    "Kalidium foliatum",
    "Limonium aureum",
    "Medicago lupulina",
    "Glycyrrhiza uralensis",
    "Dysphania schraderiana",
    "Cirsium arvense var. setosum",
    "Rorippa palustris",
    "Malva verticillata",
    "Medicago sativa",
    "Abutilon theophrasti",
    "Lycium ruthenicum",
    "Bromus tectorum",
    "Lepidium perfoliatum",
    "Corethrodendron multijugum",
    "Taraxacum mongolicum",
    "Lolium perenne",
    "Agrostis hookeriana",
    "Corispermum squarrosum",
    "Portulaca oleracea",
    "Deyeuxia flavens",
    "Potentilla potaninii",
    "Chloris virgata",
    "Incarvillea sinensis",
    "Glycyrrhiza glabra",
    "Bromus staintonii",
    "Alhagi camelorum",
    "Aristida adscensionis",
    "Trifolium repens",
    "Artemisia ordosica",
    "Plantago asiatica",
    "Puccinellia tenuiflora",
    "Poa annua",
    "Asterothamnus centraliasiaticus",
    "Salsola tragus",
    "Salsola ikonnikovii",
    "Thermopsis lanceolata",
    "Spodiopogon sibiricus",
    "Corispermum mongolicum",
    "Convolvulus arvensis",
    "Achnatherum sibiricum",
    "Polygonum aviculare",
    "Lactuca serriola",
    "Echinochloa crus-galli var. mitis",
    "Juncus effusus",
    "Melilotus albus",
    "Arctium lappa",
    "Allium mongolicum",
    "Halogeton arachnoideus",
    "Calystegia hederacea",
    "Stipa baicalensis",
    "Suaeda stellatiflora",
    "Enneapogon desvauxii",
    "Artemisia blepharolepis",
    "Lespedeza potaninii",
    "Calligonum leucocladum",
    "Elymus burchan-buddae",
    "Lespedeza davurica",
    "Lolium multiflorum",
    "Clematis fruticosa",
    "Vicia cracca",
    "Buddleja alternifolia",
    "Cynanchum acutum subsp. sibiricum",
    "Elymus pendulinus subsp. pubicaulis",
    "Calligonum mongolicum",
    "Rumex nepalensis",
    "Caragana korshinskii",
    "Pedicularis cheilanthifolia",
    "Silene aprica",
    "Senecio vulgaris",
    "Lepidium apetalum",
    "Pedicularis kansuensis",
    "Securigera varia",
    "Pennisetum × sinese",
    "Cichorium intybus",
    "Agrostis stolonifera",
    "Zoysia japonica",
    "Latouchea fokienensis",
    "Zoysia sinica",
    "Picea asperata",
    "Corethrodendron lignosum var. laeve",
    "Sorghum bicolor cv.Dochna",
    "Hordeum vulgare",
    "Cynodon dactylon",
    "Tagetes erecta",
    "Caragana microphylla",
    "× Triticosecale Wittmack",
    "Psathyrostachys juncea",
    "Hippophae rhamnoides",
    "Agropyron desertorum",
    "Agropyron mongolicum",
    "Agriophyllum pungens",
    "Cosmos bipinnatus",
    "Uraria crinita",
    "Pennisetum purpureum",
    "Rhus chinensis",
    "Puccinellia distans",
    "Astragalus sinicus",
    "Amorpha fruticosa",
    "Festuca rubra",
    "Trifolium incarnatum",
    "Leymus chinensis",
    "Festuca arundinacea",
    "Sorghum sudanense",
    "Ixeris polycephala",
    "Poa pratensis",
    "Corethrodendron fruticosum var. mongolicum",
    "Euchlaena mexicana",
    "Zoysia matrella",
    "Agropyron elongatum",
    "Avena strigosa",
    "Teloxys aristata",
    "Dracocephalum moldavica",
    "Patrinia rupestris",
    "Catolobus pendulus",
    "Odontites vulgaris",
    "Cleistogenes squarrosa",
    "Tribulus terrestris",
    "Corispermum hyssopifolium",
    "Corispermum declinatum",
    "Xanthium strumarium",
    "Artemisia sieversiana",
    "Ephedra sinica",
    "Pennisetum flaccidum",
    "Anthoxanthum glabrum",
    "Amaranthus blitum",
    "Lipschitzia divaricata",
    "Adenophora gmelinii",
    "Speranskia tuberculata",
    "Cynanchum thesioides",
    "Cirsium japonicum",
    "Asparagus cochinchinensis",
    "Belamcanda chinensis",
    "Crepidiastrum sonchifolium",
    "Inula salicina",
    "Aeluropus sinensis",
    "Amethystea caerulea",
    "Echinochloa oryzoides",
    "Klasea centauroides",
    "Polygonatum odoratum",
    "Anemarrhena asphodeloides",
    "Thalictrum simplex",
    "Solanum villosum",
    "Lespedeza juncea",
    "Chenopodium stenophyllum",
    "Iris tenuifolia",
    "Miscanthus sacchariflorus",
    "Galium verum",
    "Arundinella hirta",
    "Cannabis sativa",
    "Saussurea amara",
    "Rhamnus davurica",
    "Solanum nigrum",
    "Dendrolobium triangulare",
    "Grona heterocarpa subsp. ovalifolia",
    "Codoriocalyx gyroides",
    "Stylosanthes guianensis cv. Reyan No.21",
    "Crotalaria pallida",
    "Sesbania cannabina",
    "Macrotyloma uniflorum",
    "Polhillides velutina",
    "Desmodium intortum",
    "Tadehagi triquetrum",
    "Desmodium uncinatum",
    "Alysicarpus vaginalis",
    "Crotalaria micans",
    "Paspalum urvillei",
    "Paspalum conjugatum",
    "Galactia tenuiflora",
    "Bouffordia dichotoma",
    "Cymbopogon nardus",
    "Zenia insignis",
    "Crotalaria ferruginea",
    "Grona heterocarpos",
    "Indigofera galegoides",
    "Urochloa reptans var. glabra",
    "Crotalaria trichotoma",
    "Senna tora",
    "Sesbania bispinosa",
    "Arundinella setosa",
    "Flemingia prostrata",
    "Sohmaea zonata",
    "Dendrolobium lanceolatum",
    "Desmodium tortuosum",
    "Medicago polymorpha",
    "Paspalum distichum",
    "Dichanthium annulatum",
    "Senna bicapsularis",
    "Sporobolus diandrus",
    "Chloris formosana",
    "Aeschynomene indica",
    "Crotalaria retusa",
    "Chamaecrista mimosoides",
    "Crotalaria albida",
    "Crotalaria tetragona",
    "Chamaecrista rotundifolia",
    "Dunbaria punctata",
    "Paspalum scrobiculatum var. orbiculare",
    "Stylosanthes guianensis",
    "Panicum maximum",
    "Indigofera ramulosissima",
    "Lespedeza floribunda",
    "Flemingia macrophylla",
    "Pleurolobus gangeticus",
    "Stylosanthes macrocephala",
    "Macroptilium lathyroides",
    "Sesbania grandiflora",
    "Paspalum Longifolium",
    "Paspalum virgatum",
    "Pycnospora lutescens",
    "Piptanthus nepalensis",
    "Indigofera caudata",
    "Desmodium cinereum",
    "Grona heterophylla",
    "Panicum notatum",
    "Lespedeza cuneata",
    "Bromus catharticus",
    "Lablab purpureus",
    "Cymbopogon tortilis",
    "Phyllodium pulchellum",
    "Grona reticulata",
    "Stylosanthes hamata",
    "Senna occidentalis",
    "Indigofera tinctoria",
    "Cajanus cajan",
    "Cymbopogon citratus",
    "Crotalaria incana",
    "Panicum elegantissimum",
    "Crotalaria bracteata",
    "Paspalum dilatatum",
    "Calopogonium mucunoides",
    "Apluda mutica",
    "Puhuaea megaphylla",
    "Stylosanthes seabrana",
    "Tephrosia purpurea",
    "Neustanthus phaseoloides",
    "Eleusine indica",
    "Uraria lagopodioides",
    "Flemingia strobilifera",
    "Tephrosia candida",
    "Lespedeza cyrtobotrya",
    "Stylosanthes humilis",
    "Tephrosia pumila",
    "Arundinella nepalensis",
    "Capillipedium assimile",
    "Indigofera hirsuta",
    "Eleusine coracana",
    "Stylosanthes viscosa",
    "Paspalum plicatulum",
    "Melinis minutiflora",
    "Grona styracifolia",
    "Panicum bisulcatum",
    "Macroptilium atropurpureum",
    "Crotalaria linifolia",
    "Capillipedium parviflorum",
    "Ischaemum ciliare",
    "Vigna radiata",
    "Chamaecrista nictitans",
    "Flemingia lineata var. glutinosa",
    "Brachiaria eruciformis",
    "Sophora flavescens",
    "Beckmannia syzigachne",
    "Pueraria montana var. lobata",
    "Tadehagi pseudotriquetrum",
    "Panicum incomtum",
    "Eremochloa ciliaris",
    "Christia vespertilionis",
    "Clitoria ternatea",
    "Cullen corylifolium",
    "Vigna unguiculata",
    "Vigna umbellata",
    "Centotheca lappacea",
    "Glycine soja",
    "Lilium brownii",
    "Eriochloa villosa",
    "Vachellia farnesiana",
    "Crotalaria uncinella",
    "Lespedeza pilosa",
    "Leucaena leucocephala",
    "Indigofera hendecaphylla",
    "Crotalaria lanceolata",
    "Dunbaria podocarpa",
    "Puhuaea sequax",
    "Christia constricta",
    "Paspalum thunbergii",
    "Urochloa paspaloides",
    "Panicum maximum var. trichoglume",
    "Cymbopogon mekongensis",
    "Andropogon munroi",
    "Stylosanthes hippocampoides",
    "Indigofera bungeana",
    "Eriochloa procera",
    "Kummerowia striata",
    "Dunbaria truncata",
    "Rhynchosia volubilis",
    "Senna surattensis",
    "Paspalum atratum",
    "Sporobolus fertilis",
    "Senecio nemorensis",
    "Chenopodium ficifolium",
    "Bidens pilosa",
    "Hedysarum gmelinii",
    "Delphinium pylzowii",
    "Bupleurum longiradiatum",
    "Picris hieracioides",
    "Adenophora stricta",
    "Salsola collina",
    "Dracocephalum heterophyllum",
    "Thymus mongolicus",
    "Braya humilis",
    "Crepis rigescens",
    "Siphonostegia chinensis",
    "Saussurea japonica",
    "Festuca dahurica",
    "Scabiosa comosa",
    "Artemisia frigida",
    "Dontostemon micranthus",
    "Allium anisopodium",
    "Neopallasia pectinata",
    "Euphorbia esula",
    "Leptopyrum fumarioides",
    "Thalictrum squarrosum",
    "Agropyron michnoi",
    "Butomus umbellatus",
    "Ajuga linearifolia",
    "Aster alpinus",
    "Pulsatilla turczaninovii",
    "Oxytropis myriophylla",
    "Ligularia mongolica",
    "Delphinium grandiflorum",
    "Artemisia dracunculus",
    "Artemisia annua",
    "Thesium chinense",
    "Halerpestes sarmentosa",
    "Iris tigridia",
    "Cynoglossum divaricatum",
    "Thalictrum petaloideum",
    "Lathyrus quinquenervius",
    "Lycopus lucidus",
    "Bothriospermum chinense",
    "Epilobium hirsutum",
    "Clematis hexapetala",
    "Poa sphondylodes",
    "Bolboschoenus yagara",
    "Takhtajaniantha austriaca",
    "Argentina anserina",
    "Androsace filiformis",
    "Allium tenuissimum",
    "Phlomoides tuberosa",
    "Artemisia scoparia",
    "Chenopodium acuminatum",
    "Allium condensatum",
    "Pseudolysimachion incanum",
    "Chamaerhodos erecta",
    "Artemisia palustris",
    "Atriplex centralasiatica var. megalotheca",
    "Clematis tangutica",
    "Kalidium gracile",
    "Milium effusum",
    "Neotrinia splendens",
    "Ranunculus chinensis",
    "Artemisia capillaris",
    "Geranium sibiricum",
    "Kalidium cuspidatum",
    "Caroxylon passerinum",
    "Lotus corniculatus",
    "Aster tataricus",
    "Cuscuta chinensis",
    "Cynoglossum amabile",
    "Cirsium arvense var. integrifolium",
    "Acanthocalyx nepalensis",
    "Deschampsia cespitosa",
    "Thalictrum alpinum",
    "Leymus angustus",
    "Astragalus densiflorus",
    "Anisodus tanguticus",
    "Vicia unijuga",
    "Heracleum hemsleyanum",
    "Oxytropis kansuensis",
    "Dianthus superbus",
    "Trollius farreri",
    "Dicranostigma leptopodum",
    "Gentiana macrophylla",
    "Brassica juncea",
    "Medicago ruthenica",
    "Halenia corniculata",
    "Anemone rivularis",
    "Cirsium souliei",
    "Cnidium monnieri",
    "Axyris amaranthoides",
    "Codonopsis pilosula",
    "Gentiana straminea",
    "Oxytropis ochrocephala",
    "Rheum pumilum",
    "Carex parva",
    "Carex microglochin",
    "Stellera chamaejasme",
    "Bistorta vivipara",
    "Caragana brevifolia",
    "Kengyilia thoroldiana",
    "Gentiana macrophylla f. alba",
    "Anemone cathayensis",
    "Gentiana siphonantha",
    "Carex capillifolia",
    "Saussurea pulchra",
    "Corydalis conspersa",
    "Astragalus yunnanensis",
    "Oxytropis falcata",
    "Przewalskia tangutica",
    "Iris tectorum",
    "Carex pseudosupina",
    "Festuca ovina",
    "Oxytropis proboscidea",
    "Potentilla saundersiana",
    "Leontopodium pusillum",
    "Artemisia selengensis",
    "Incarvillea younghusbandii",
    "Stipa purpurea",
    "Carex atrofusca",
    "Oxytropis microphylla",
    "Carex parvula",
    "Arctopoa tibetica",
    "Festuca tibetica",
    "Artemisia younghusbandii",
    "Pedicularis tibetica",
    "Astragalus tibetanus",
    "Astragalus oxyglottis",
    "Corispermum lehmannianum",
    "Isatis gymnocarpa",
    "Lactuca undulata",
    "Heterocaryum rigidum",
    "Tetracme quadricornis",
    "Hyoscyamus pusillus",
    "Garhadiolus papposus",
    "Diptychocarpus strictus",
    "Trigonella arcuata",
    "Goldbachia Laevigata",
    "Chorispora sibirica",
    "Cancrinia discoidea",
    "Strigosella scorpioides",
    "Meniocus linifolius",
    "Crepis desertorum",
    "Centaurea pulchella",
    "Zygophyllum macropterum",
    "Zygophyllum lehmannianum",
    "Lallemantia royleana",
    "Lepidium ruderale",
    "Lepidium chalepense",
    "Pseudosedum lievenii",
    "Strigosella africana",
    "Zygophyllum pterocarpum",
    "Stipagrostis pennata",
    "Ferula sibirica",
    "Erodium oxyrhinchum",
    "Tetracme recurvata",
    "Eremurus inderiensis",
    "Euclidium syriacum",
    "Turgenia latifolia",
    "Hedysarum dahuricum",
    "Koelpinia linearis",
    "Cuminum borszczowii",
    "Allium caeruleum",
    "Salvia deserta",
    "Rudolf-kamelinia korolkowii",
    "Marrubium vulgare",
    "Camelina microcarpa",
    "Melica transsilvanica",
    "Trollius dschungaricus",
    "Phlomoides pratensis",
    "Draba nemorosa",
    "Camelina sativa",
    "Iljinskaea planisiliqua",
    "Alyssum simplex",
    "Avena fatua",
    "Lepidium latifolium",
    "Dodartia orientalis",
    "Leontopodium leontopodioides",
    "Veronica ciliata",
    "Aconitum karakolicum",
    "Chamerion angustifolium",
    "Silene vulgaris",
    "Climacoptera obtusifolia",
    "Horaninovia ulicina",
    "Heteracia szovitsii",
    "Sisymbrium altissimum",
    "Haloxylon persicum",
    "Euphorbia inderiensis",
    "Aconitum flavum",
    "Papaver nudicaule",
    "Ziziphus jujuba var. spinosa",
    "Allium senescens",
    "Agrimonia pilosa",
    "Potentilla tanacetifolia",
    "Aster hispidus",
    "Stipa capillata",
    "Panicum miliaceum",
    "Clematis intricata",
    "Viola prionantha",
    "Elsholtzia ciliata",
    "Androsace septentrionalis",
    "Cyclachaena xanthiifolia",
    "Oxytropis racemosa",
    "Digitaria sanguinalis",
    "Flueggea suffruticosa",
    "Jacobaea ambracea",
    "Carex duriuscula",
    "Allium macrostemon",
    "Inula japonica",
    "Dianthus chinensis",
    "Astragalus tenuis",
    "Sonchus wightianus",
    "Hordeum brevisubulatum",
    "Vicia sepium",
    "Clematis florida",
    "Allium neriniflorum",
    "Dactylis glomerata",
    "Artemisia gmelinii var. messerschmidiana",
    "Euryops pectinatus",
]


## Define Functions

In [ ]:
def find_best_location(
    target_size: tuple[int, int],
    img_size: tuple[int, int],
    occupied: list[tuple[int, int, int, int]],
    overlap_threshold: float = 0.1,
    min_distance: float = 0.0,
    circle_like: bool = False,
    max_circle_r: float = 1.0,
) -> tuple[int, int, int, int] | None:
    target_w, target_h = target_size
    img_w, img_h = img_size

    if target_w <= 0 or target_h <= 0:
        return None

    if img_w - target_w <= 0 or img_h - target_h <= 0:
        return None

    def calculate_overlap_ratio(
        rect1: tuple[int, int, int, int],
        rect2: tuple[int, int, int, int],
    ) -> float:
        x1, y1, w1, h1 = rect1
        x2, y2, w2, h2 = rect2

        left = max(x1, x2)
        top = max(y1, y2)
        right = min(x1 + w1, x2 + w2)
        bottom = min(y1 + h1, y2 + h2)

        if left >= right or top >= bottom:
            return 0.0

        overlap_area = (right - left) * (bottom - top)

        area = max(0, min(w1 * h1, w2 * h2))

        return overlap_area / area if area > 0 else 0.0

    def calculate_distance(
        rect1: tuple[int, int, int, int],
        rect2: tuple[int, int, int, int],
    ) -> float:
        x1, y1, w1, h1 = rect1
        x2, y2, w2, h2 = rect2

        cx1, cy1 = x1 + w1 // 2, y1 + h1 // 2
        cx2, cy2 = x2 + w2 // 2, y2 + h2 // 2

        dist = math.sqrt((cx1 - cx2) ** 2 + (cy1 - cy2) ** 2)

        return dist

    def calculate_circle_distance(rect: tuple[int, int, int, int]) -> float:
        x, y, w, h = rect
        # right, bottom
        r, b = x + w, y + h
        # image center
        cx, cy = img_w / 2, img_h / 2
        dx = cx - x if x < cx else r - cx
        dy = cy - y if y < cy else b - cy
        dist = math.sqrt(dx**2 + dy**2) / math.sqrt(cx**2 + cy**2)
        return dist

    def is_valid_position(x: int, y: int) -> bool:
        candidate_rect = (x, y, target_w, target_h)

        for occupied_rect in occupied:
            overlap_ratio = calculate_overlap_ratio(candidate_rect, occupied_rect)
            if overlap_ratio > overlap_threshold:
                return False
            distance = calculate_distance(candidate_rect, occupied_rect)
            if distance < min_distance:
                return False
            if circle_like:
                circle_distance = calculate_circle_distance(candidate_rect)
                if circle_distance > max_circle_r:
                    return False
        return True

    candidates: list[tuple[float, int, int, int, int]] = []

    step_size = max(1, min(target_w, target_h) // 4)

    for y in range(0, img_h - target_h + 1, step_size):
        for x in range(0, img_w - target_w + 1, step_size):
            if is_valid_position(x, y):
                center_x, center_y = img_w // 2, img_h // 2
                rect_center_x, rect_center_y = x + target_w // 2, y + target_h // 2
                distance_to_center: float = math.sqrt(
                    (rect_center_x - center_x) ** 2 + (rect_center_y - center_y) ** 2
                )
                score = -distance_to_center
                candidates.append((score, x, y, target_w, target_h))

    if not candidates:
        margin = max(target_w, target_h) // 2

        for y in range(-margin, img_h + margin, step_size):
            for x in range(-margin, img_w + margin, step_size):
                if 0 <= x <= img_w - target_w and 0 <= y <= img_h - target_h:
                    continue

                if is_valid_position(x, y):
                    overlap_left = max(0, x)
                    overlap_top = max(0, y)
                    overlap_right = min(img_w, x + target_w)
                    overlap_bottom = min(img_h, y + target_h)

                    if overlap_right > overlap_left and overlap_bottom > overlap_top:
                        overlap_area = (overlap_right - overlap_left) * (overlap_bottom - overlap_top)
                        score = overlap_area
                        candidates.append(
                            (
                                score,
                                x,
                                y,
                                overlap_right - overlap_left,
                                overlap_bottom - overlap_top,
                            )
                        )

    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        _, x, y, w, h = candidates[0]
        return (x, y, w, h) if w > 0 and h > 0 else None

    return None

In [ ]:
def gen_one_image(
    image_set: list[str],
    label_set: list[int],
    num_rect_min: int = NUM_RECTS_MIN,
    num_rect_max: int = NUM_RECTS_MAX,
    draw_rect: bool = False,
    img_size: tuple[int, int] = (2192, 2192),
    max_overlap: float = 0.1,
    rect_dist_range: tuple[float, float] = (0.0, 300.0),
    circle_r_range: tuple[float, float] = (0.7, 1.0),
) -> tuple[Image.Image, list[dict[str, int | float | list[float]]], np.ndarray]:
    assert len(image_set) == len(label_set)

    num_rect = random.randint(num_rect_min, num_rect_max)

    # background
    bg: str | SyntheticBg = random.choice(SYNTHETIC_BACKGROUNDS)
    if isinstance(bg, str):
        canvas = Image.new("RGB", img_size, bg)
    elif isinstance(bg, SyntheticBg):
        canvas = Image.open(bg.path)
        canvas = canvas.resize(img_size, resample=Image.Resampling.BILINEAR)
    else:
        raise ValueError("Invalid background")

    # background transform
    transpose_ops = [
        -1,
        Image.Transpose.FLIP_LEFT_RIGHT,
        Image.Transpose.FLIP_TOP_BOTTOM,
        Image.Transpose.ROTATE_90,
        Image.Transpose.ROTATE_180,
        Image.Transpose.ROTATE_270,
    ]
    op = random.choice(transpose_ops)
    if op != -1 and isinstance(op, Image.Transpose):
        canvas = canvas.transpose(op)

    # background mask, image type is i16, since the class labels are in
    # [0, 656], u8 is not enough, -1 means background
    canvas_mask = np.zeros(canvas.size, dtype=np.int16) - 1

    occupied_rects: list[tuple[int, int, int, int]] = []
    annotations: list[dict[str, int | float | list[float]]] = []

    # random selected image indexes
    idxs: NDArray[np.int32] = np.random.choice(
        np.arange(len(image_set)),
        size=num_rect,
        replace=True,
        p=sampling_p_of(label_set),
    ).astype(np.int32)
    for idx in idxs:
        seed_img_name = SRC_IMAGES_DIR / image_set[idx]
        seed_img_bg_name = SRC_IMAGES_WITH_BG_DIR / image_set[idx]
        seed_mask_name = SRC_MASKS_DIR / image_set[idx]
        seed_label = label_set[idx]

        seed_mask = Image.open(seed_mask_name).convert("L")
        seed_img_with_bg = Image.open(seed_img_bg_name)
        seed_img = Image.open(seed_img_name)
        angle = random.randint(0, 360)
        # print(f"rotate {angle} degrees for {seed_img_name}")
        seed_img = seed_img.rotate(angle, expand=True)
        bbox = seed_img.getbbox()
        seed_img = seed_img.crop(bbox)  # remove blank area
        seed_mask = seed_mask.rotate(angle, expand=True)
        seed_mask = seed_mask.crop(bbox)  # remove blank area, keep same size with seed_img
        seed_img_with_bg = seed_img_with_bg.rotate(angle, expand=True)
        seed_img_with_bg = seed_img_with_bg.crop(bbox)  # remove blank area
        seed_img = Image.composite(seed_img_with_bg, Image.new("RGB", seed_img.size, (0, 0, 0)), seed_mask)
        op = random.choice(transpose_ops[:3])  # already rotated, ignore rotation OPs
        if op != -1 and isinstance(op, Image.Transpose):
            seed_img = seed_img.transpose(op)
            seed_mask = seed_mask.transpose(op)
        assert seed_img.size == seed_mask.size, f"{seed_img.size=} != {seed_mask.size=}"

        best_rect = find_best_location(
            seed_img.size,
            canvas.size,
            occupied_rects,
            overlap_threshold=random.uniform(0.0, max_overlap),
            min_distance=random.uniform(rect_dist_range[0], rect_dist_range[1]),
            circle_like=False if isinstance(bg, str) else bg.circle_like,
            max_circle_r=random.uniform(circle_r_range[0], circle_r_range[1]),
        )
        if best_rect is None:
            continue
        try:
            x, y, w, h = best_rect
            canvas.paste(seed_img, (x, y), seed_mask)

            contours, _ = cv.findContours(
                np.asarray(seed_mask, dtype=np.uint8),
                cv.RETR_EXTERNAL,
                cv.CHAIN_APPROX_SIMPLE,
            )
            if len(contours) == 0:
                continue
            contour = contours[0].copy().reshape(-1, 2).astype(np.float32)
            contour[:, 0] = (contour[:, 0] + x) / canvas_mask.shape[1]  # width
            contour[:, 1] = (contour[:, 1] + y) / canvas_mask.shape[0]  # height
            np.swapaxes(contour, 0, 1)  # x, y -> y, x

            # make background_mask
            _mask = np.asarray(seed_mask, dtype=np.int16).T
            assert _mask.ndim == 2
            _mask[_mask <= 0] = -1
            _mask[_mask > 0] = seed_label
            w1, h1 = _mask.shape

            canvas_mask[x : x + w1, y : y + h1] = _mask

            if draw_rect:
                draw = ImageDraw.Draw(canvas)
                draw.rectangle([(x, y), (x + w, y + h)], outline=random_color(), width=2)

            occupied_rects.append(best_rect)
            annotations.append(
                {
                    "label": seed_label,
                    "x": x / canvas.width,
                    "y": y / canvas.height,
                    "w": w / canvas.width,
                    "h": h / canvas.height,
                    "coordinates": contour.flatten().tolist(),
                }
            )
        except Exception as e:
            # can not find matched area for the current seed_image, skip it.
            print(f"error: {e}")
            continue

    return canvas, annotations, canvas_mask


def to_yolo(
    labels: list[tuple[int, float, float, float, float]],
) -> list[tuple[float, float, float, float, float]]:
    detect: list[tuple[float, float, float, float, float]] = []
    for d in labels:
        label, x, y, w, h = d
        cx, cy = (x + w / 2), (y + h / 2)
        detect.append((label, round(cx, 6), round(cy, 6), round(w, 6), round(h, 6)))
    return detect

## Generate one to test

In [ ]:
img, annos, mask = gen_one_image(
    TRAIN_IMAGES,
    TRAIN_LABELS,
    num_rect_max=20,
    draw_rect=True,
)
img.save(f"{0}.png")
np.savez_compressed(f"{0}_mask.npz", mask)
coco_detect = [
    (
        d["label"],
        # YOLO format, center_x, center_y, w, h
        # https://docs.ultralytics.com/datasets/detect/
        round(d["x"] + d["w"] / 2, 6),  # type: ignore
        round(d["y"] + d["h"] / 2, 6),  # type: ignore
        round(d["w"], 6),  # type: ignore
        round(d["h"], 6),  # type: ignore
    )
    for d in annos
]
coco_seg = [[d["label"], *[round(x, 6) for x in d["coordinates"]]] for d in annos]  # type: ignore
# annos_detect, annos_seg = to_coco(annos, mask)
detect_str = "\n".join(
    [
        " ".join([str(x) if isinstance(x, int) else f"{x:.6f}" for x in d])  #
        for d in coco_detect
    ]
)
seg_str = "\n".join(
    [
        " ".join([str(x) if isinstance(x, int) else f"{x:.6f}" for x in d])  #
        for d in coco_seg
    ]
)
with open(f"{0}.txt", "w+", encoding="utf-8") as fout:
    fout.write(detect_str)
with open(f"{0}_seg.txt", "w+", encoding="utf-8") as fout:
    fout.write(seg_str)
with open("anno.json", "w") as f:
    json.dump(annos, f)

## batch generate

In [ ]:
def gen_func(job_id: int, subset: Literal["train", "val", "test"]):
    tgt_img_path = DST_IMAGES_DIR / subset / f"{job_id}.png"
    tgt_mask_img_path = DST_MASKS_IMG_DIR / subset / f"{job_id}.npz"
    tgt_label_path = DST_LABELS_DIR / subset / f"{job_id}.txt"
    tgt_mask_txt_path = DST_MASKS_DIR / subset / f"{job_id}.txt"
    tgt_anno_path = DST_ANNO_DIR / subset / f"{job_id}.json"
    if (
        tgt_img_path.exists()
        and tgt_mask_img_path.exists()
        and tgt_label_path.exists()
        and tgt_mask_txt_path.exists()
        and tgt_anno_path.exists()
    ):
        print(f"Skipping {job_id}")
        return job_id, tgt_img_path.relative_to(DST_IMAGES_DIR)

    tgt_img_path.parent.mkdir(parents=True, exist_ok=True)
    tgt_mask_img_path.parent.mkdir(parents=True, exist_ok=True)
    tgt_label_path.parent.mkdir(parents=True, exist_ok=True)
    tgt_mask_txt_path.parent.mkdir(parents=True, exist_ok=True)
    tgt_anno_path.parent.mkdir(parents=True, exist_ok=True)

    if subset == "train":
        img, annos, mask = gen_one_image(TRAIN_IMAGES, TRAIN_LABELS)
    elif subset == "val":
        img, annos, mask = gen_one_image(VAL_IMAGES, VAL_LABELS)
    elif subset == "test":
        img, annos, mask = gen_one_image(TEST_IMAGES, TEST_LABELS)
    else:
        raise ValueError(f"Unknown subset: {subset}")
    img.save(tgt_img_path)
    np.savez_compressed(tgt_mask_img_path, mask)

    coco_detect = [
        (
            d["label"],
            # YOLO format, center_x, center_y, w, h
            # https://docs.ultralytics.com/datasets/detect/
            round(d["x"] + d["w"] / 2, 6),  # type: ignore
            round(d["y"] + d["h"] / 2, 6),  # type: ignore
            round(d["w"], 6),  # type: ignore
            round(d["h"], 6),  # type: ignore
        )
        for d in annos
    ]
    coco_seg = [[d["label"], *[round(x, 6) for x in d["coordinates"]]] for d in annos]  # type: ignore
    detect_str = "\n".join(
        [
            " ".join([str(x) if isinstance(x, int) else f"{x:.6f}" for x in d])  #
            for d in coco_detect
        ]
    )
    seg_str = "\n".join(
        [
            " ".join([str(x) if isinstance(x, int) else f"{x:.6f}" for x in d])  #
            for d in coco_seg
        ]
    )
    with open(tgt_label_path, "w", encoding="utf-8") as fout:
        fout.write(detect_str)
    with open(tgt_mask_txt_path, "w", encoding="utf-8") as fout:
        fout.write(seg_str)
    with open(tgt_anno_path, "w", encoding="utf-8") as fout:
        json.dump(annos, fout, ensure_ascii=False)

    return job_id, tgt_img_path.relative_to(DST_IMAGES_DIR)


n_train = int(DATASET_SIZE * P_TRAIN)
n_val = int(DATASET_SIZE * P_VAL)
n_test = DATASET_SIZE - n_train - n_val
for subset, start_idx, subset_size in [
    ("train", 0, n_train),
    ("val", n_train, n_val),
    ("test", n_train + n_val, n_test),
]:
    image_paths = []
    res = Parallel(n_jobs=N_JOBS)(
        delayed(gen_func)(i, subset) for i in range(start_idx, start_idx + subset_size)
    )
    if res is None:
        raise ValueError("res is None")
    for job_id, img_path in res:  # type: ignore
        image_paths.append(str(img_path))
    with open(DST_DIR / f"{subset}.txt", "w", encoding="utf-8") as fout:
        fout.write("\n".join(image_paths))
